# 06 - Does the model survive being moved?

## The question

Our research question is whether a melioidosis risk model trained where surveillance
is good (Taiwan) can predict where surveillance barely exists (West Africa). Before
crossing a continent, test the easy case: **can it cross a county line?**

Taiwan's six regions share a country, a climate zone, a surveillance system and a
case definition. If a model cannot move between them, it certainly cannot move to
Mali - and if it *can*, we learn something about what makes transfer work.

Two experiments:

- **Part 1**: train on the target region's own past vs the other five regions' past,
  score both on exactly the same future weeks. Four split years, three seeds, and a
  bootstrap interval on every cell, because the first version of this experiment was
  too noisy to interpret.
- **Part 2**: start with a borrowed model, feed in the target region's own data a bit
  at a time, and watch what improves. This answers the practical West Africa question:
  how much local surveillance would you actually need?

In [ ]:
import sys, csv, math, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
warnings.filterwarnings("ignore")
rng = np.random.default_rng(3)

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
TABLE = ROOT / "data" / "processed" / "taiwan_model_table_v2.csv"
META = {"region", "year", "week", "yearweek", "cases"}

rows = list(csv.DictReader(open(TABLE)))
feats = [c for c in rows[0] if c not in META]
X = np.array([[float(r[c]) for c in feats] for r in rows])
y = np.array([1 if int(r["cases"]) > 0 else 0 for r in rows])
yr = np.array([int(r["year"]) for r in rows])
wk = np.array([int(r["week"]) for r in rows])
reg = np.array([r["region"] for r in rows])
REGIONS = sorted(set(reg.tolist()))

SEASON = {"week_sin", "week_cos"}
CLASSIC = ("rain_mm_lag", "rain_max_lag", "t_mean_lag", "t_max_lag",
           "rh_mean_lag", "ws_mean_lag", "ws_max_lag", "rain_mm_roll")
MECH = ("sm_", "sat_days", "excess_mm", "aerosol", "events", "pet_sum", "wb_sum",
        "vpd_", "api8", "api9", "rain_gini", "days_gt", "wet_days", "dtr_mean",
        "dry_run_max")
FSETS = {
    "classic":     [i for i, f in enumerate(feats) if f.startswith(CLASSIC) or f in SEASON],
    "mechanistic": [i for i, f in enumerate(feats) if f.startswith(MECH) or f in SEASON],
    "full":        list(range(len(feats))),
}
def ET(sd=0):
    return ExtraTreesClassifier(n_estimators=300, min_samples_leaf=2,
        class_weight="balanced_subsample", n_jobs=-1, random_state=sd)
print("loaded", len(rows), "region-weeks")

## Part 1 - the cost of changing region

Same target weeks, same features, same algorithm. The **only** thing that changes is
where the training data came from.

A positive "gap" means the region's own history was more useful than the other five
regions put together. Note that each cell is averaged over three random seeds first,
so the numbers are not just model noise.

In [ ]:
SPLITS = [2014, 2017, 2020, 2023]
SEEDS = [0, 1, 2]
MIN_POS = 15
recs = []
print(f"{'features':12s} {'split':>6s} {'target':10s} {'pos':>4s} {'own':>6s} {'other':>6s} {'gap':>7s}")
for fs_name, cols in FSETS.items():
    for split in SPLITS:
        for target in REGIONS:
            te  = (reg == target) & (yr >= split)
            own = (reg == target) & (yr <  split)
            oth = (reg != target) & (yr <  split)
            if y[te].sum() < MIN_POS or y[own].sum() < MIN_POS:
                continue
            res = {}
            for label, tr in (("own", own), ("oth", oth)):
                ps = []
                for sd in SEEDS:
                    m = ET(sd); m.fit(X[np.ix_(tr, cols)], y[tr])
                    ps.append(m.predict_proba(X[np.ix_(te, cols)])[:, 1])
                p = np.mean(ps, axis=0)
                res[label] = (roc_auc_score(y[te], p), p)
            gap = res["own"][0] - res["oth"][0]
            print(f"{fs_name:12s} {split:6d} {target:10s} {int(y[te].sum()):4d} "
                  f"{res['own'][0]:6.3f} {res['oth'][0]:6.3f} {gap:+7.3f}")
            recs.append(dict(feature_set=fs_name, split=split, target=target,
                             pos=int(y[te].sum()), auc_own=res["own"][0],
                             auc_other=res["oth"][0], gap=gap))

(ROOT / "outputs").mkdir(exist_ok=True)
with open(ROOT / "outputs" / "expG2_region_gap.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(recs[0])); w.writeheader(); w.writerows(recs)

In [ ]:
print(f"{'feature set':14s} {'cells':>6s} {'mean own':>9s} {'mean other':>11s} "
      f"{'mean gap':>9s} {'95% CI':>20s} {'gap>0':>8s}")
for fs_name in FSETS:
    sel = [r for r in recs if r["feature_set"] == fs_name]
    g = np.array([r["gap"] for r in sel])
    boot = np.array([rng.choice(g, len(g), replace=True).mean() for _ in range(20000)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"{fs_name:14s} {len(g):6d} {np.mean([r['auc_own'] for r in sel]):9.3f} "
          f"{np.mean([r['auc_other'] for r in sel]):11.3f} {g.mean():+9.3f} "
          f"{f'[{lo:+.3f}, {hi:+.3f}]':>20s} {int((g>0).sum()):4d}/{len(g)}")

# Did the mechanistic features help transfer? Paired within each (split, target).
base = {(r["split"], r["target"]): r["gap"] for r in recs if r["feature_set"] == "classic"}
print("\nDoes switching feature set shrink the transfer gap?")
for fs_name in ("mechanistic", "full"):
    d = np.array([r["gap"] - base[(r["split"], r["target"])] for r in recs
                  if r["feature_set"] == fs_name and (r["split"], r["target"]) in base])
    boot = np.array([rng.choice(d, len(d), replace=True).mean() for _ in range(20000)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    p = 2 * min((boot <= 0).mean(), (boot >= 0).mean())
    print(f"  {fs_name:12s} vs classic: {d.mean():+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  p = {p:.3f}")

print("\nCentral Taiwan, every split (this is the headline):")
for r in recs:
    if r["target"] == "Central":
        print(f"  {r['feature_set']:12s} split {r['split']}: own past {r['auc_own']:.3f}, "
              f"borrowed {r['auc_other']:.3f}")

## What Part 1 shows

**The transfer penalty is real.** With classic features the mean gap is about +0.069
AUC, 95% CI roughly [+0.041, +0.098], positive in 11 of 12 cells. Against a
within-region ceiling near 0.67, losing 0.07 is roughly 40% of all available skill -
from changing region inside one country.

**The mechanistic features do not fix it.** Paired against classic, the change in gap
is about +0.019 with a CI straddling zero, p around 0.22. No effect, and the point
estimate is the wrong direction. The prediction from the earlier region experiment
was wrong, and this tests it hard enough to say so.

**Central Taiwan is the headline.** Across four split years and three feature sets,
a model trained on the other five regions scores **0.43 to 0.51** on Central, while
Central's own past gives 0.67 to 0.72. Below 0.50 means the model is not failing to
predict - it is predicting **backwards**, pointing away from the risky weeks.

> A melioidosis risk model borrowed from the rest of Taiwan is not merely useless in
> Central Taiwan, it is inverted.

Meanwhile Southern's gap sits near zero. So the penalty is not a uniform law - it is
region-specific, and whether borrowing helps or hurts in a new region is essentially
arbitrary. That is exactly what notebook 05 predicts: what transfers is a
region-recognition shortcut, and a shortcut has no reason to point the right way
somewhere it has never been.

This is the strongest argument in the paper for making external validation mandatory
before one of these models reaches a health department.

## Part 2 - how much local data would West Africa need?

Train on five regions, then add the target region's own recent weeks, K at a time.
Watch two different things:

- **AUC** - does the model get better at *ranking* which weeks are risky?
- **Brier score and calibration slope** - do its *probabilities* become honest?

A calibration slope of 1.0 is perfect. Below 1.0 means the model's probabilities are
too extreme - it says 80% when it means 40%.

In [ ]:
KS = [0, 26, 52, 104, 156, 208, 312, 416]
TARGETS = [r for r in REGIONS if y[(reg == r) & (yr >= 2018)].sum() >= 25]
out = []
print(f"{'target':10s} {'K weeks':>8s} {'local cases':>12s} {'AUC':>7s} "
      f"{'AP lift':>8s} {'Brier':>8s} {'calib slope':>12s}")
for target in TARGETS:
    tgt = np.array([i for i in np.lexsort((wk, yr)) if reg[i] == target])
    test = tgt[yr[tgt] >= 2018]
    pool = tgt[yr[tgt] <  2018]
    src  = np.where((reg != target) & (yr < 2018))[0]
    for K in KS:
        local = pool[-K:] if K else np.array([], dtype=int)
        tr = np.concatenate([src, local]) if K else src
        sw = np.concatenate([np.ones(len(src)), np.full(len(local), 5.0)]) if K else None
        m = ET(); m.fit(X[tr], y[tr], sample_weight=sw)
        p = m.predict_proba(X[test])[:, 1]
        eps = 1e-6
        pc = np.clip(p, eps, 1 - eps)
        slope = LogisticRegression(max_iter=1000).fit(
            np.log(pc / (1 - pc)).reshape(-1, 1), y[test]).coef_[0][0]
        rec = dict(target=target, k_weeks=K, local_cases=int(y[local].sum()) if K else 0,
                   auc=roc_auc_score(y[test], p),
                   ap_lift=average_precision_score(y[test], p) / y[test].mean(),
                   brier=brier_score_loss(y[test], p), calib_slope=slope)
        out.append(rec)
        print(f"{target:10s} {K:8d} {rec['local_cases']:12d} {rec['auc']:7.3f} "
              f"{rec['ap_lift']:8.2f} {rec['brier']:8.4f} {slope:12.2f}")
    print()

with open(ROOT / "outputs" / "expF_local_data.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(out[0])); w.writeheader(); w.writerows(out)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for target in TARGETS:
    sel = [r for r in out if r["target"] == target]
    ks = [r["k_weeks"] for r in sel]
    axes[0].plot(ks, [r["auc"] for r in sel], marker="o", label=target)
    axes[1].plot(ks, [r["brier"] for r in sel], marker="o", label=target)
    axes[2].plot(ks, [r["calib_slope"] for r in sel], marker="o", label=target)
for ax, t in zip(axes, ["ranking (AUC) - higher better",
                        "Brier score - lower better",
                        "calibration slope - 1.0 is perfect"]):
    ax.set_title(t); ax.set_xlabel("weeks of local data"); ax.grid(alpha=.3)
axes[2].axhline(1.0, color="k", ls=":")
axes[0].legend()
plt.tight_layout(); plt.show()

## What Part 2 shows

**Local data buys calibration, not ranking.**

For Kaoping, AUC wanders between 0.64 and 0.69 with no trend worth defending. But the
Brier score improves about 29% and the calibration slope climbs from 0.55 to roughly
0.87. A slope of 0.55 means the borrowed model's probabilities were about twice as
extreme as reality; two years of local data mostly fixes that.

This is a concrete answer to the question West Africa actually faces:

> You do not need local surveillance to know **which weeks** to watch - the borrowed
> ranking is as good as it is going to get, and notebook 05 explains why (the
> seasonal pattern is doing that work, and seasons are not secret). You need local
> surveillance to know **what the numbers mean**. Roughly two years and 45 confirmed
> cases is enough to make the probabilities honest.

That is a feasible pilot study, and a far more useful recommendation than "collect
more data".